In [9]:
# from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .appName("Building Gold Medallion Workflow") \
#     .config("spark.jars", "C:/spark_jar/postgresql-42.7.13.jar") \
#     .getOrCreate()

# print(spark.sparkContext.getConf().get('spark.jars'))

import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-25.0.4.101-hotspot"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Building Gold Medallion Workflow") \
    .config(
        "spark.jars",
        r"C:\spark_jar\postgresql-42.7.13.jar"
    ) \
    .getOrCreate()


print("Spark:", spark.version)

Spark: 4.2.0


In [10]:
customers_silver = spark.read.parquet("../Silver/silver/customers")
orders_silver = spark.read.parquet("../Silver/silver/orders")
order_items_silver = spark.read.parquet("../Silver/silver/order_items")
order_payments_silver = spark.read.parquet("../Silver/silver/order_payments")
order_reviews_silver = spark.read.parquet("../Silver/silver/order_reviews")
sellers_silver = spark.read.parquet("../Silver/silver/sellers")
products_silver = spark.read.parquet("../Silver/silver/products")
category_silver = spark.read.parquet("../Silver/silver/.category")
geolocation_silver = spark.read.parquet("../Silver/silver/geolocation")

In [11]:
for old_col in geolocation_silver.columns:
    new_col = old_col.strip('"')
    geolocation_silver = geolocation_silver.withColumnRenamed(old_col, new_col)

print(geolocation_silver.columns)  # confirm clean now

['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


In [12]:
from pyspark.sql import functions as F
geolocation_dedup = geolocation_silver.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("avg_lat"),
    F.avg("geolocation_lng").alias("avg_lng")
)

In [13]:
dim_customer = customers_silver.select(
    "customer_id", "customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"
).dropDuplicates(["customer_id"]) \
 .join(geolocation_dedup, customers_silver["customer_zip_code_prefix"] == geolocation_dedup["geolocation_zip_code_prefix"], "left") \
 .select("customer_id", "customer_unique_id", "customer_city", "customer_state", "avg_lat", "avg_lng") \
 .withColumn("customer_key", F.monotonically_increasing_id())

In [14]:
for old_col in sellers_silver.columns:
    new_col = old_col.strip('"')
    sellers_silver = sellers_silver.withColumnRenamed(old_col, new_col)

print(sellers_silver.columns)  # confirm clean now

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [15]:
dim_seller = sellers_silver.select(
    "seller_id", "seller_city", "seller_state", "seller_zip_code_prefix"
).dropDuplicates(["seller_id"]) \
 .join(geolocation_dedup, sellers_silver["seller_zip_code_prefix"] == geolocation_dedup["geolocation_zip_code_prefix"], "left") \
 .select("seller_id", "seller_city", "seller_state", "avg_lat", "avg_lng") \
 .withColumn("seller_key", F.monotonically_increasing_id())

In [16]:
for old_col in products_silver.columns:
    new_col = old_col.strip('"')
    products_silver = products_silver.withColumnRenamed(old_col, new_col)

print(products_silver.columns)  # confirm clean now

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [17]:
dim_product = products_silver.join(
    category_silver, products_silver["product_category_name"] == category_silver["product_category_name"], "left"
).select(
    products_silver["product_id"],
    category_silver["product_category_name_english"].alias("product_category")
).dropDuplicates(["product_id"]) \
 .withColumn("product_key", F.monotonically_increasing_id())

In [18]:
dim_date = orders_silver.select(
    F.to_date("order_purchase_timestamp").alias("order_date")
).dropDuplicates(["order_date"]) \
 .withColumn("date_key", F.monotonically_increasing_id()) \
 .withColumn("day_of_week", F.date_format("order_date", "EEEE")) \
 .withColumn("is_weekend", F.dayofweek("order_date").isin([1, 7]))

In [19]:
fact_orders = orders_silver \
    .join(order_items_silver, "order_id", "inner") \
    .join(order_payments_silver, "order_id", "inner") \
    .join(order_reviews_silver, "order_id", "left") \
    .join(dim_customer, "customer_id", "inner") \
    .join(dim_product, "product_id", "inner") \
    .join(dim_seller, "seller_id", "inner") \
    .join(dim_date, orders_silver["order_purchase_timestamp"].cast("date") == dim_date["order_date"], "inner") \
    .select(
        "order_id",
        dim_customer["customer_key"],
        dim_product["product_key"],
        dim_seller["seller_key"],
        dim_date["date_key"],
        "price",
        "freight_value",
        "payment_value",
        "review_score"
    )

In [20]:
def write_gold_to_postgres(df, table_name):
    df.write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://localhost:5432/E_Commerce") \
        .option("dbtable", table_name) \
        .option("user", "postgres") \
        .option("password", "Kickboxing12#") \
        .option("driver", "org.postgresql.Driver") \
        .mode("overwrite") \
        .save()
    print(f"{table_name} written to Postgres")

write_gold_to_postgres(dim_customer, "dim_customer")
write_gold_to_postgres(dim_seller, "dim_seller")
write_gold_to_postgres(dim_product, "dim_product")
write_gold_to_postgres(dim_date, "dim_date")
write_gold_to_postgres(fact_orders, "fact_orders")

dim_customer written to Postgres
dim_seller written to Postgres
dim_product written to Postgres
dim_date written to Postgres
fact_orders written to Postgres
